# 31 — Collect SOTA Experiment Results

Run this after the individual experiment notebooks. It loads the result pickles and prints a compact comparison table.


In [ ]:
import pickle
from pathlib import Path

result_files = {
    # ── Non-learned baselines ─────────────────────────────────────────────
    "pca":               "/tmp/results_sota_pca_wj_512.pkl",
    "random_proj":       "/tmp/results_sota_random_proj_wj_512.pkl",
    "nmf":               "/tmp/results_sota_nmf_wj_512.pkl",
    "icws":              "/tmp/results_sota_icws_512.pkl",

    # ── Neural baselines ──────────────────────────────────────────────────
    "autoencoder":        "/tmp/results_sota_autoencoder_wj_512.pkl",
    "triplet_autoencoder":"/tmp/results_sota_triplet_autoencoder_wj_512.pkl",
    "deep_binary_hash":  "/tmp/results_sota_deep_binary_hash_wj_512.pkl",
    "matryoshka":        "/tmp/results_sota_matryoshka_mlp_wj_512.pkl",

    # ── Our MLP variants ──────────────────────────────────────────────────
    "mlp_cosine_512":         "/tmp/results_mlp_cosine.pkl",
    "mlp_intersection_min_512":"/tmp/results_mlp_intersection_min.pkl",
    "mlp_two_stage_wj":       "/tmp/results_two_stage_mlp_wj.pkl",
    "mlp_wj_native_no_fn":    "/tmp/results_mlp_wj_native_no_fn_filter.pkl",
    "mlp_wj_reg_maxpos100":   "/tmp/results_mlp_wj_reg_maxpos100.pkl",
}

# Runs to skip within a pickle (stale/wrong-method entries)
SKIP_RUNS = {
    "mlp_cosine_512": {"bhattacharyya"},
}

# For matryoshka: only show the full 512-dim run (d512), skip sub-dim variants
MATRYOSHKA_KEEP_DIM = "d512"

dataset_name = "10k"
sections = {"No Rerank": [], "Rerank 500": [], "Rerank 1000": []}

def section_for_run(name, metrics):
    ck = metrics.get("candidate_k")
    if "no_rerank" in name: return "No Rerank"
    if ck == 500  or "k500" in name or name.endswith("_rerank_500"):  return "Rerank 500"
    if ck == 1000 or "k1000" in name or name.endswith("_rerank_1000"): return "Rerank 1000"
    if "rerank" in name: return None
    return "No Rerank"

GROUP_LABELS = {
    "pca":                "PCA simplex 512",
    "random_proj":        "Random Projection 512",
    "nmf":                "NMF simplex 512",
    "icws":               "ICWS WeightedMinHash 512  [brute-force]",
    "autoencoder":        "AE reconstruction 512",
    "triplet_autoencoder":"AE triplet+recon 512 (nb28)",
    "deep_binary_hash":   "Deep Binary Hashing 512",
    "matryoshka":         "Matryoshka MLP 512",
    "mlp_cosine_512":     "MLP cosine 512",
    "mlp_intersection_min_512": "MLP intersection-min 512",
    "mlp_two_stage_wj":   "MLP two-stage WJ",
    "mlp_wj_native_no_fn":"MLP WJ-triplet (baseline)",
    "mlp_wj_reg_maxpos100":"MLP WJ-triplet+reg (best)",
}

for group, path in result_files.items():
    p = Path(path)
    if not p.exists():
        print(f"  [missing] {GROUP_LABELS.get(group, group)}: {path}")
        continue
    with open(p, "rb") as f:
        payload = pickle.load(f)
    runs = payload.get(dataset_name, {})
    skip = SKIP_RUNS.get(group, set())
    for name, metrics in runs.items():
        if name in skip:
            continue
        # For matryoshka: skip sub-dim variants, keep only d512
        if group == "matryoshka" and f"_{MATRYOSHKA_KEEP_DIM}" not in name:
            continue
        section = section_for_run(name, metrics)
        if section is None:
            continue
        label = GROUP_LABELS.get(group, group)
        if len(runs) > 1 and section != "No Rerank":
            ck = metrics.get("candidate_k", "?")
            label = f"{label} [K={ck}]"
        sections[section].append((
            label,
            metrics.get(10, float("nan")),
            metrics.get(50, float("nan")),
            metrics.get(100, float("nan")),
            metrics.get(500, float("nan")),
            metrics.get("qps", float("nan")),
        ))

def print_section(title, rows, icws_note=False):
    print(f"\n{'='*80}")
    print(f"  {title}")
    if icws_note:
        print(f"  (ICWS excluded — brute-force O(N), cannot build ANN index)")
    print(f"{'='*80}")
    if not rows: print("  (no results yet)"); return
    print(f"  {'Method':<55} {'R@10':>7} {'R@50':>7} {'R@100':>7} {'R@500':>7} {'QPS':>9}")
    print(f"  {'-'*98}")
    for row in sorted(rows, key=lambda x: (-(x[2] if x[2] == x[2] else -1))):
        name, r10, r50, r100, r500, qps = row
        print(f"  {name:<55} {r10:>7.4f} {r50:>7.4f} {r100:>7.4f} {r500:>7.4f} {qps:>9.1f}")

print_section("No Rerank", sections["No Rerank"])
print_section("Rerank 500",  sections["Rerank 500"],  icws_note=True)
print_section("Rerank 1000", sections["Rerank 1000"], icws_note=True)
